# Trend Dashboard Metrics

This notebook prepares trend metrics for the Streamlit dashboard.

## Features:
1. **Multi-horizon growth rates** (2 weeks, 1 month, 3 months, 6 months, 9 months, 1 year)
2. **Granularity levels**: Global (all malls) and per-mall
3. **Category breakdown**: BL1 (high-level) and BL2 (detailed) categories
4. **Forecasted growth rates** at 1, 2, and 3 month horizons

## 1. Imports and Setup

In [ ]:
from datetime import timedelta
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
# Import project constants
import constants.constants as cst
import constants.paths as pth

## 2. Data Loading

In [ ]:
# Load dimension tables
dim_blocks = pd.read_csv(pth.DIM_BLOCKS, **cst.CSV_PARAMS)
dim_malls = pd.read_csv(pth.DIM_MALLS, **cst.CSV_PARAMS)

# Load fact tables
print("Loading fact_stores...")
fact_stores = pd.read_csv(pth.FACT_STORES, **cst.CSV_PARAMS)

print(f"fact_stores shape: {fact_stores.shape}")
print(f"dim_blocks shape: {dim_blocks.shape}")
print(f"dim_malls shape: {dim_malls.shape}")

In [ ]:
# Convert date and sort
fact_stores["date"] = pd.to_datetime(fact_stores["date"], format="%d/%m/%Y")
fact_stores = fact_stores.sort_values("date")

# Enrich with category and mall info
stores_enriched = fact_stores.merge(
    dim_blocks[["block_id", "bl1_label", "bl2_label"]],
    on="block_id",
    how="left",
)

stores_enriched = stores_enriched.merge(
    dim_malls[["id", "country", "mall_name"]],
    left_on="mall_id",
    right_on="id",
    how="left",
)

# Fill missing mall names with mall_id
stores_enriched["mall_name"] = stores_enriched["mall_name"].fillna(
    "Mall_" + stores_enriched["mall_id"].astype(str)
)

# Data overview
date_min = stores_enriched["date"].min()
date_max = stores_enriched["date"].max()
print(f"\nDate range: {date_min.date()} to {date_max.date()}")
print(f"Number of malls: {stores_enriched['mall_id'].nunique()}")
print(f"Number of stores: {stores_enriched['store_code'].nunique()}")

## 3. Define Time Horizons

In [ ]:
# Time horizons for growth rate calculation (in days)
TIME_HORIZONS = {
    "2_weeks": 14,
    "1_month": 30,
    "3_months": 90,
    "6_months": 180,
    "9_months": 270,
    "1_year": 365,
}

# Forecast horizons
FORECAST_HORIZONS = {
    "1_month": 30,
    "2_months": 60,
    "3_months": 90,
}

# Reference date (latest date in data)
REFERENCE_DATE = stores_enriched["date"].max()
print(f"Reference date for calculations: {REFERENCE_DATE.date()}")

## 4. Growth Rate Calculation Functions

In [ ]:
def calculate_growth_rate(
    df: pd.DataFrame,
    metric: str = "people_in",
    horizon_days: int = 30,
    reference_date: pd.Timestamp = None,
) -> dict:
    """Calculate growth rate for a given time horizon.

    Args:
        df: DataFrame with 'date' and metric columns
        metric: Column name to calculate growth for
        horizon_days: Number of days to look back
        reference_date: Date to calculate from (default: max date in df)

    Returns:
        dict with growth rate info or None if insufficient data
    """
    if reference_date is None:
        reference_date = df["date"].max()

    # Define periods
    current_start = reference_date - timedelta(days=horizon_days)
    previous_start = current_start - timedelta(days=horizon_days)

    # Get available data range
    data_start = df["date"].min()

    # Adjust if we don't have enough historical data
    actual_horizon = min(horizon_days, (reference_date - data_start).days // 2)

    if actual_horizon < 7:  # Minimum 1 week of data required
        return None

    # Recalculate periods with actual horizon
    current_start = reference_date - timedelta(days=actual_horizon)
    previous_start = current_start - timedelta(days=actual_horizon)

    # Current period data
    current_data = df[(df["date"] > current_start) & (df["date"] <= reference_date)]
    current_value = current_data[metric].sum()

    # Previous period data
    previous_data = df[(df["date"] > previous_start) & (df["date"] <= current_start)]
    previous_value = previous_data[metric].sum()

    # Calculate growth rate
    if previous_value == 0:
        growth_rate = np.nan
    else:
        growth_rate = ((current_value - previous_value) / previous_value) * 100

    return {
        "current_value": current_value,
        "previous_value": previous_value,
        "growth_rate": growth_rate,
        "actual_horizon_days": actual_horizon,
        "requested_horizon_days": horizon_days,
        "data_sufficient": actual_horizon == horizon_days,
    }

In [ ]:
def calculate_multi_horizon_growth(
    df: pd.DataFrame,
    group_col: str = None,
    metric: str = "people_in",
    reference_date: pd.Timestamp = None,
) -> pd.DataFrame:
    """Calculate growth rates across all time horizons.

    Args:
        df: DataFrame with traffic data
        group_col: Column to group by (e.g., 'bl1_label', 'mall_name')
        metric: Metric to calculate growth for
        reference_date: Date to calculate from

    Returns:
        DataFrame with growth rates for each horizon
    """
    if reference_date is None:
        reference_date = df["date"].max()

    results = []

    # Get groups to iterate over
    if group_col:
        groups = df[group_col].dropna().unique()
    else:
        groups = ["Global"]

    for group in groups:
        if group_col:
            group_df = df[df[group_col] == group]
        else:
            group_df = df

        row = {"group": group}

        for horizon_name, horizon_days in TIME_HORIZONS.items():
            growth_info = calculate_growth_rate(
                group_df, metric, horizon_days, reference_date
            )

            if growth_info:
                row[f"{horizon_name}_growth"] = growth_info["growth_rate"]
                row[f"{horizon_name}_sufficient"] = growth_info["data_sufficient"]
            else:
                row[f"{horizon_name}_growth"] = np.nan
                row[f"{horizon_name}_sufficient"] = False

        results.append(row)

    return pd.DataFrame(results)

In [ ]:
def classify_trend(growth_rate: float) -> str:
    """Classify trend based on growth rate."""
    if pd.isna(growth_rate):
        return "Insufficient Data"
    elif growth_rate > 10:
        return "Strongly Emerging"
    elif growth_rate > 2:
        return "Emerging"
    elif growth_rate > -2:
        return "Stable"
    elif growth_rate > -10:
        return "Declining"
    else:
        return "Strongly Declining"


def calculate_mall_category_growth(
    df: pd.DataFrame,
    category_col: str = "bl1_label",
) -> pd.DataFrame:
    """Calculate growth rates for each mall-category combination."""
    results = []

    for mall in df["mall_name"].dropna().unique():
        mall_df = df[df["mall_name"] == mall]

        for category in mall_df[category_col].dropna().unique():
            cat_df = mall_df[mall_df[category_col] == category]

            # Aggregate by date
            daily = cat_df.groupby("date").agg({"people_in": "sum"}).reset_index()

            row = {"mall": mall, "category": category}

            for horizon_name, horizon_days in TIME_HORIZONS.items():
                growth_info = calculate_growth_rate(
                    daily, "people_in", horizon_days, REFERENCE_DATE
                )

                if growth_info:
                    row[f"{horizon_name}_growth"] = growth_info["growth_rate"]
                    row[f"{horizon_name}_sufficient"] = growth_info["data_sufficient"]
                else:
                    row[f"{horizon_name}_growth"] = np.nan
                    row[f"{horizon_name}_sufficient"] = False

            results.append(row)

    return pd.DataFrame(results)

## 5. Global Level - Growth Rates by Category (All Malls)

Global level aggregates traffic across all malls, broken down by category.

In [ ]:
# Aggregate daily traffic by BL1 category
daily_by_bl1 = (
    stores_enriched.groupby(["date", "bl1_label"])
    .agg({"people_in": "sum"})
    .reset_index()
)

# Calculate growth rates for each BL1 category
bl1_growth = calculate_multi_horizon_growth(daily_by_bl1, group_col="bl1_label")

# Add trend classifications
for horizon in TIME_HORIZONS.keys():
    bl1_growth[f"{horizon}_trend"] = bl1_growth[f"{horizon}_growth"].apply(
        classify_trend
    )

print("BL1 Category Growth Rates (1-month horizon):")
print("=" * 60)
display_cols = ["group", "1_month_growth", "1_month_trend", "3_months_growth"]
print(
    bl1_growth[display_cols]
    .sort_values("1_month_growth", ascending=False)
    .to_string(index=False)
)

In [ ]:
# Visualize BL1 growth rates across horizons
growth_cols = [col for col in bl1_growth.columns if col.endswith("_growth")]
horizons_display = [col.replace("_growth", "") for col in growth_cols]

fig = go.Figure()

for _, row in bl1_growth.iterrows():
    values = [row[col] for col in growth_cols]
    fig.add_trace(
        go.Scatter(
            x=horizons_display,
            y=values,
            mode="lines+markers",
            name=row["group"],
        )
    )

fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="BL1 Category Growth Rates by Time Horizon",
    xaxis_title="Time Horizon",
    yaxis_title="Growth Rate (%)",
    height=500,
)
fig.show()

## 6. Mall Level - Growth Rates by Category (Per Mall)

Breaks down growth rates by category within each mall.

In [ ]:
# Calculate growth rates for each mall × category combination
mall_category_growth = calculate_mall_category_growth(stores_enriched, "bl1_label")

# Add trend classifications for 1-month horizon
mall_category_growth["1_month_trend"] = mall_category_growth["1_month_growth"].apply(
    classify_trend
)

print("Mall × Category Growth Rates (1-month horizon):")
print("=" * 60)
display_cols = ["mall", "category", "1_month_growth", "1_month_trend"]
print(
    mall_category_growth[display_cols]
    .sort_values(["mall", "1_month_growth"], ascending=[True, False])
    .head(30)
    .to_string(index=False)
)

In [ ]:
# Heatmap: Select a category and view all malls' growth rates
selected_category = "Fashion apparel"  # Change this to explore different categories

category_data = mall_category_growth[
    mall_category_growth["category"] == selected_category
].copy()

if len(category_data) > 0:
    growth_cols = [col for col in category_data.columns if col.endswith("_growth")]
    heatmap_data = category_data.set_index("mall")[growth_cols]
    heatmap_data.columns = [col.replace("_growth", "") for col in growth_cols]

    fig = px.imshow(
        heatmap_data,
        labels=dict(x="Time Horizon", y="Mall", color="Growth Rate (%)"),
        title=f"Mall Growth Rates for '{selected_category}' by Time Horizon",
        color_continuous_scale="RdYlGn",
        color_continuous_midpoint=0,
        aspect="auto",
    )
    fig.update_layout(height=600)
    fig.show()
else:
    print(f"No data for category: {selected_category}")

## 7. Forecasting Growth Rates

In [ ]:
# Load Prophet
try:
    from prophet import Prophet

    prophet_available = True
    print("Prophet loaded successfully.")
except ImportError:
    prophet_available = False
    print("Prophet not available. Install with: pip install prophet")

In [ ]:
def forecast_growth_rates(
    df: pd.DataFrame,
    metric: str = "people_in",
    min_history_days: int = 60,
) -> dict:
    """Forecast growth rates at 1, 2, and 3 month horizons.

    Args:
        df: DataFrame with 'date' and metric columns (daily aggregated)
        metric: Column to forecast
        min_history_days: Minimum days of history required for forecasting

    Returns:
        dict with forecasted growth rates for each horizon
    """
    if not prophet_available:
        return {"error": "Prophet not available"}

    # Check if we have enough data
    data_days = (df["date"].max() - df["date"].min()).days
    if data_days < min_history_days:
        return {
            "error": f"Insufficient data: {data_days} days (need {min_history_days})"
        }

    # Prepare data for Prophet
    prophet_df = df[["date", metric]].copy()
    prophet_df.columns = ["ds", "y"]
    prophet_df = prophet_df.dropna()

    if len(prophet_df) < 30:
        return {"error": "Insufficient non-null data points"}

    # Fit model (suppress output)
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
    )
    model.fit(prophet_df)

    # Make predictions
    max_horizon = max(FORECAST_HORIZONS.values())
    future = model.make_future_dataframe(periods=max_horizon)
    forecast = model.predict(future)

    # Calculate forecasted growth rates
    current_value = prophet_df["y"].iloc[-30:].sum()  # Last 30 days
    reference_date = df["date"].max()

    results = {"current_30d_value": current_value}

    for horizon_name, horizon_days in FORECAST_HORIZONS.items():
        # Get forecasted values for the horizon period
        forecast_start = reference_date + timedelta(days=1)
        forecast_end = reference_date + timedelta(days=horizon_days)

        forecast_period = forecast[
            (forecast["ds"] >= forecast_start) & (forecast["ds"] <= forecast_end)
        ]

        if len(forecast_period) > 0:
            # Sum over the forecast period (equivalent to horizon_days)
            forecast_value = forecast_period["yhat"].sum()

            # Normalize to compare same-length periods
            normalized_current = current_value * (horizon_days / 30)

            if normalized_current > 0:
                growth_rate = (
                    (forecast_value - normalized_current) / normalized_current
                ) * 100
            else:
                growth_rate = np.nan

            results[f"{horizon_name}_forecast"] = forecast_value
            results[f"{horizon_name}_growth"] = growth_rate
            results[f"{horizon_name}_trend"] = classify_trend(growth_rate)
        else:
            results[f"{horizon_name}_forecast"] = np.nan
            results[f"{horizon_name}_growth"] = np.nan
            results[f"{horizon_name}_trend"] = "Insufficient Data"

    return results

In [ ]:
# Quick test: Forecast for a sample category
if prophet_available:
    # Test with one category first
    test_category = stores_enriched["bl1_label"].dropna().iloc[0]
    print(f"Testing forecast for: {test_category}")
    print("=" * 60)

    test_daily = (
        stores_enriched[stores_enriched["bl1_label"] == test_category]
        .groupby("date")
        .agg({"people_in": "sum"})
        .reset_index()
    )

    result = forecast_growth_rates(test_daily)

    if "error" not in result:
        for horizon in FORECAST_HORIZONS.keys():
            rate = result.get(f"{horizon}_growth")
            trend = result.get(f"{horizon}_trend")
            if pd.notna(rate):
                print(f"  {horizon}: {rate:+.2f}% [{trend}]")
            else:
                print(f"  {horizon}: Insufficient data")
    else:
        print(f"Error: {result['error']}")

In [ ]:
# Forecast by BL1 category (all categories)
if prophet_available:
    print("Forecasting growth rates for ALL BL1 categories...")
    print("=" * 60)

    bl1_forecasts = []

    for category in stores_enriched["bl1_label"].dropna().unique():
        # Prepare category data
        cat_daily = (
            stores_enriched[stores_enriched["bl1_label"] == category]
            .groupby("date")
            .agg({"people_in": "sum"})
            .reset_index()
        )

        result = forecast_growth_rates(cat_daily)
        result["category"] = category
        bl1_forecasts.append(result)

    # Create forecast summary DataFrame
    forecast_summary = pd.DataFrame(bl1_forecasts)

    # Display results
    print("\nForecast Summary by Category:")
    for fc in bl1_forecasts:
        cat = fc["category"]
        if "error" in fc:
            print(f"\n{cat}: {fc['error']}")
        else:
            print(f"\n{cat}:")
            for horizon in FORECAST_HORIZONS.keys():
                rate = fc.get(f"{horizon}_growth")
                trend = fc.get(f"{horizon}_trend")
                if pd.notna(rate):
                    print(f"  {horizon}: {rate:+.2f}% [{trend}]")
                else:
                    print(f"  {horizon}: Insufficient data")

## 8. Summary Tables for Streamlit

In [ ]:
# Create summary dataframes that can be exported for Streamlit

# 1. Global Level: BL1 Category Summary (all malls aggregated)
bl1_summary = bl1_growth.copy()
bl1_summary = bl1_summary.rename(columns={"group": "category"})

# 2. Mall Level: Mall × Category detailed
mall_category_summary = mall_category_growth.copy()

print("Summary Tables Created:")
print(f"  - bl1_summary: {len(bl1_summary)} categories (global level)")
print(f"  - mall_category_summary: {len(mall_category_summary)} mall-category pairs")

In [ ]:
# Preview BL1 summary
print("BL1 Category Summary:")
display_cols = [
    "category",
    "2_weeks_growth",
    "1_month_growth",
    "3_months_growth",
    "6_months_growth",
    "1_month_trend",
]
bl1_summary[display_cols].sort_values("1_month_growth", ascending=False)

In [ ]:
# Preview Mall × Category summary
print("Mall × Category Summary (sample):")
display_cols = [
    "mall",
    "category",
    "2_weeks_growth",
    "1_month_growth",
    "3_months_growth",
    "1_month_trend",
]
mall_category_summary["1_month_trend"] = mall_category_summary["1_month_growth"].apply(
    classify_trend
)
mall_category_summary[display_cols].sort_values(
    ["mall", "1_month_growth"], ascending=[True, False]
).head(20)

## Next Steps for Streamlit Integration

The following data structures are ready for export:

### Data Tables:
1. **`bl1_summary`**: Global-level growth rates by category (all malls aggregated)
2. **`mall_category_summary`**: Mall-level growth rates by category (per-mall breakdown)
3. **`forecast_summary`**: Forecasted growth rates at 1/2/3 month horizons

### Key Functions to Port:
- `calculate_growth_rate()`: Core growth calculation with adaptive horizon
- `calculate_multi_horizon_growth()`: Multi-horizon wrapper
- `calculate_mall_category_growth()`: Mall × category calculations
- `classify_trend()`: Trend classification

- `forecast_growth_rates()`: Prophet-based forecasting- **Forecast View**: 1/2/3 month predictions

- **Time Horizon Selector**: 2 weeks → 1 year

### Dashboard Features:- **Mall View**: Filter by mall to see category performance
- **Global View**: Category trends across all malls